# Steady-state predictive analysis
This notebook reproduce the results related to steady state approaches used in the article. 

These models are:  
- Gradient Boosting (GB)
- Multi-layer Perceptron (MLP)

In [ ]:
import sys
sys.path.append("../.")
import os
import numpy as np
from xgboost import XGBRegressor

from OpenDeckGeneration.data_cv import create_kfold_splits, create_flatten_datasets
from OpenDeckGeneration.evaluator import evaluate_predictions, extract_metrics_for_table
from package.data import Data
from package.plots.plots import plot_cv_losses

In [2]:
file_name = "2000_aligned_and_clean_indexfalse.csv"
file_path = os.path.join("..", "data", file_name)

data = Data()
data.load(load_path=file_path)

### Gradient Boosting
Herein, we train and evaluate through 5 folds cross validation a XGB regressor to learn the health indicators from sensor measurements.

In [ ]:
predictions_all = []
observations_all = []
feature_importances = []
metrics = {}

for train_idx, val_idx, test_idx in create_kfold_splits(n_sequences=len(data.sequences),
                                                        k_folds=5,
                                                        train_ratio=0.7,
                                                        val_ratio=0.1,
                                                        test_ratio=0.2):

    train_dataset, val_dataset, test_dataset, scaler = create_flatten_datasets(data.sequences, 
                                                                               train_idx, 
                                                                               val_idx, 
                                                                               test_idx, 
                                                                               shuffle_train=True,
                                                                               scale_inputs=True)
    train_sensors, train_indicators = train_dataset
    val_sensors, val_indicators = val_dataset
    test_sensors, test_indicators = test_dataset
    
    
    regr = XGBRegressor(n_estimators=100, max_depth=50)
    regr.fit(train_sensors,
             train_indicators)
    predictions = regr.predict(test_sensors)
    predictions_all.append(predictions)
    observations_all.append(test_indicators)
    feature_importances.append(regr.feature_importances_)
    evaluate_predictions(y_true=test_indicators, y_pred=predictions, metrics=metrics)       

Show the evaluation metrics and extract those one reported in the article.

In [ ]:
metric_dict = extract_metrics_for_table(metrics)

### MLP
Herein, we train and evaluate through 5 folds cross validation a MLP model to learn the health indicators from sensor measurements.

In [ ]:
import torch
from package.model.architecture.mlp import MLP
from package.model.mlp_trainer import MLPTrainer

In [ ]:
predictions_all = []
observations_all = []
train_losses = []
val_losses = []
device = "cuda:0"
metrics = {}

for train_idx, val_idx, test_idx in create_kfold_splits(n_sequences=len(data.sequences),
                                                        k_folds=5,
                                                        train_ratio=0.7,
                                                        val_ratio=0.1,
                                                        test_ratio=0.2):
    train_loader, val_loader, test_loader, scaler = data.create_loader_flatten(train_idx,
                                                                               val_idx,
                                                                               test_idx,
                                                                               batch_size=128,
                                                                               shuffle_train=True,
                                                                               scale_inputs=True,
                                                                               scale_outputs=True)
    model = MLP(input_size=28, 
                hidden_sizes=(192, 224, 224), 
                output_size=10, 
                device=device, 
                dropout=2e-4)
    model.to(float)
    optimizer = torch.optim.Adam(model.parameters(), lr=6e-5, weight_decay=7.9e-6)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',        # because we monitor validation loss
        factor=0.2,        # reduce LR by 1/5
        patience=3,        # wait 3 epochs before reducing
    )

    trainer = MLPTrainer(model, 
                        lr=6e-5, 
                        optimizer=optimizer,
                        scheduler=None,
                        device=device, scaler=scaler)
    trainer.train_model(train_loader=train_loader, val_loader=val_loader, epochs=100)
    train_losses.append(trainer.train_losses)
    val_losses.append(trainer.val_losses)
    observations, predictions = trainer.predict_loader(test_loader)
    predictions_all.append(predictions)
    observations_all.append(observations)
    evaluate_predictions(y_true=observations, y_pred=predictions, metrics=metrics)

In [ ]:
train_losses = [np.array(train_loss) for train_loss in train_losses]
val_losses = [np.array(val_loss) for val_loss in val_losses]

plot_cv_losses(training_losses=train_losses, validation_losses=val_losses)

In [ ]:
metric_dict = extract_metrics_for_table(metrics)